In [13]:
import numpy as np
import torch
import pandas as pd
import math
print("numpy version:", np.__version__)

numpy version: 1.26.4


In [2]:
pd.set_option('display.width', 10000) 

In [8]:
# create a numpy array
bs = 1
seq_len = 3
num_attention_heads = 2
attention_head_size = 5

emb_size = num_attention_heads * attention_head_size

numpyarray = (np.arange(bs * seq_len * emb_size)+1).reshape(bs, seq_len, emb_size)
# print(numpyarray)
# convert to pytorch tensor
a = torch.from_numpy(numpyarray)

print(f"bs: {bs}, seq_len: {seq_len}, num_attention_heads: {num_attention_heads}, attention_head_size: {attention_head_size}")
print(a)

bs: 1, seq_len: 3, num_attention_heads: 2, attention_head_size: 5
tensor([[[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
         [11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
         [21, 22, 23, 24, 25, 26, 27, 28, 29, 30]]])


In [9]:
b = a.view(bs, seq_len, num_attention_heads, attention_head_size)
print(f"bs: {bs}, seq_len: {seq_len}, num_attention_heads: {num_attention_heads}, attention_head_size: {attention_head_size}")

print(b)

bs: 1, seq_len: 3, num_attention_heads: 2, attention_head_size: 5
tensor([[[[ 1,  2,  3,  4,  5],
          [ 6,  7,  8,  9, 10]],

         [[11, 12, 13, 14, 15],
          [16, 17, 18, 19, 20]],

         [[21, 22, 23, 24, 25],
          [26, 27, 28, 29, 30]]]])


In [10]:
c = proj = b.transpose(1, 2)   # Swap seq_len and num_heads
print(f"bs: {bs}, seq_len: {seq_len}, num_attention_heads: {num_attention_heads}, attention_head_size: {attention_head_size}")

print(f"b shape: {b.shape}")
print(f"c shape: {c.shape}")

print(c)

bs: 1, seq_len: 3, num_attention_heads: 2, attention_head_size: 5
b shape: torch.Size([1, 3, 2, 5])
c shape: torch.Size([1, 2, 3, 5])
tensor([[[[ 1,  2,  3,  4,  5],
          [11, 12, 13, 14, 15],
          [21, 22, 23, 24, 25]],

         [[ 6,  7,  8,  9, 10],
          [16, 17, 18, 19, 20],
          [26, 27, 28, 29, 30]]]])


In [11]:
S = torch.matmul(c, c.transpose(-1, -2)) / math.sqrt(attention_head_size)
print(f"S shape: {S.shape}")
S

S shape: torch.Size([1, 2, 3, 3])


tensor([[[[  24.5967,   91.6788,  158.7608],
          [  91.6788,  382.3676,  673.0565],
          [ 158.7608,  673.0565, 1187.3521]],

         [[ 147.5805,  326.4659,  505.3513],
          [ 326.4659,  728.9581, 1131.4504],
          [ 505.3513, 1131.4504, 1757.5494]]]])

In [12]:
## Mask some values in S
attention_mask = torch.zeros((bs, seq_len, seq_len))
attention_mask[0, 0, 1] = -10000.0
attention_mask[0, 1, 2] = -10000.0
attention_mask[1, 0, 2] = -10000.0
attention_mask = attention_mask.unsqueeze(1)  # Add num_attention_heads dimension   
print(f"attention_mask shape: {attention_mask.shape}")
print(attention_mask)

# Apply the attention mask to S
print(S)
S = S + attention_mask
print(S)

IndexError: index 1 is out of bounds for dimension 0 with size 1

In [18]:
df = pd.read_csv('./data/sst-sentiment-train.csv')
df.shape

(7898, 3)

In [21]:
df.sentence.apply(len).describe()


count    7898.000000
mean      102.810712
std        50.791050
min         4.000000
25%        64.000000
50%        99.000000
75%       137.000000
max       261.000000
Name: sentence, dtype: float64

In [24]:
pd.set_option('display.max_colwidth', 200)  # Show full text in DataFrame
df[['sentence', 'sentiment']].head(10)

,sentence,sentiment
0,The drama is played out with such aching beauty and truth that it brings tears to your eyes .,4
1,"It 's hard to care about a film that proposes as epic tragedy the plight of a callow rich boy who is forced to choose between his beautiful , self-satisfied 22-year-old girlfriend and an equally b...",1
2,"The entire point of a shaggy dog story , of course , is that it goes nowhere , and this is classic nowheresville in every sense .",1
3,The sort of movie that gives tastelessness a bad rap .,0
4,Criminal conspiracies and true romances move so easily across racial and cultural lines in the film that it makes My Big Fat Greek Wedding look like an apartheid drama .,3
5,Great over-the-top moviemaking if you 're in a slap-happy mood .,4
6,"The crime matters less than the characters , although the filmmakers supply enough complications , close calls and double-crosses to satisfy us .",2
7,Everything that was right about Blade is wrong in its sequel .,1
8,"Despite Auteuil 's performance , it 's a rather listless amble down the middle of the road , where the thematic ironies are too obvious and the sexual politics too smug .",1
9,"Passions , obsessions , and loneliest dark spots are pushed to their most virtuous limits , lending the narrative an unusually surreal tone .",3


In [31]:
df.sentiment.value_counts()

sentiment
1    2108
3    2053
2    1474
4    1239
0    1024
Name: count, dtype: int64

In [41]:
df.loc[df.sentiment==0, ['sentence', 'sentiment']].head(10)

,sentence,sentiment
3,The sort of movie that gives tastelessness a bad rap .,0
10,Tries to work in the same vein as the brilliance of Animal House but instead comes closer to the failure of the third Revenge of the Nerds sequel .,0
17,"If you 're looking for comedy to be served up , better look elsewhere .",0
29,"I found it slow , predictable and not very amusing .",0
31,There 's not one decent performance from the cast and not one clever line of dialogue .,0
36,"Build some robots , haul 'em to the theatre with you for the late show , and put on your own Mystery Science Theatre 3000 tribute to what is almost certainly going to go down as the worst -- and o...",0
52,"It 's dumb , but more importantly , it 's just not scary .",0
65,"Like a bad improvisation exercise , the superficially written characters ramble on tediously about their lives , loves and the art they 're struggling to create .",0
70,Has all the scenic appeal of a cesspool .,0
73,Fred Schepisi 's film is paced at a speed that is slow to those of us in middle age and deathly slow to any teen .,0
